# ReSnap v5, langkah 2: audit kotak tumpukan dan pratinjau data rimpang

Belum ada training di sini. Dua pertanyaan yang dijawab:

1. **Apakah skor rendah brokoli, jamur, kunyit, dll. sebagian disebabkan anotasi tumpukan?**
   Dugaan dari langkah 1: ada foto yang setumpuk bahan diberi satu kotak, sementara model mendeteksi per objek.
   Diukur di val (hanya dihitung, label val tidak diubah), lalu dicari kasus yang sama di train.
2. **Seperti apa 5 dataset rimpang Roboflow?** Jumlah foto, kelas, gaya anotasi, salinan augmentasi,
   foto kembar dengan data ReSnap, dan bahan ReSnap lain yang tidak berlabel.

Kebutuhan Kaggle: GPU T4, Internet on, Input `resnap-v4`, Secret `ROBOFLOW_API_KEY`.

Keluaran:
- `audit_v5.zip`: laporan kecil, kirim ke chat.
- `rimpang_raw/*.zip`: data rimpang mentah, nanti dijadikan dataset Kaggle `resnap-v5-rimpang` supaya tidak perlu unduh ulang.

In [ ]:
%pip -q install ultralytics==8.3.228 roboflow

In [ ]:
import hashlib, json, math, random, shutil, subprocess, sys, zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image, ImageDraw, ImageOps

assert torch.cuda.is_available(), 'GPU tidak aktif. Settings > Accelerator > GPU T4.'
print('GPU:', torch.cuda.get_device_name(0))
random.seed(0)

INPUT, OUTPUT, TMP = Path('/kaggle/input'), Path('/kaggle/working'), Path('/tmp/resnap')
OUT = OUTPUT / 'audit_v5'
RAW = OUTPUT / 'rimpang_raw'
for p in (OUT, RAW, TMP):
    p.mkdir(parents=True, exist_ok=True)

REPO = TMP / 'resnap-cv'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/AffanAbya30/resnap-cv.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'scripts'))
import data_v4 as dv

V4B_SHA = '1410f11d9c2cdfad074446d1742ed5ec24d0269122bc51ec5ddde69bafc66809'
WEIGHTS = REPO / 'models/best.pt'
assert dv.sha256_file(WEIGHTS) == V4B_SHA, 'models/best.pt bukan v4b'

def first_dir(pred):
    hits = sorted((p for p in INPUT.rglob('*') if p.is_dir() and pred(p)), key=lambda p: len(p.parts))
    return hits[0] if hits else None

DATA = first_dir(lambda p: (p / 'holdout_oi/images').is_dir() and (p / 'train/images').is_dir())
BASE = first_dir(lambda p: (p / 'test/images').is_dir() and (p / 'train/images').is_dir())
assert DATA, 'Folder data v4 (punya holdout_oi) tidak ditemukan di Input resnap-v4'
print('Data v4:', DATA, '| data v3 (untuk cek foto kembar):', BASE)

from ultralytics import YOLO
model = YOLO(str(WEIGHTS))
NAMES = model.names
IDX = {v: k for k, v in NAMES.items()}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def images(folder):
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMG_EXT and '_dup' not in p.stem)

def read_labels(lbl_path):
    """YOLO bbox atau poligon -> (cls, x1, y1, x2, y2) relatif."""
    out = []
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            v = line.split()
            if len(v) < 5:
                continue
            c, nums = int(v[0]), list(map(float, v[1:]))
            if len(nums) == 4:
                cx, cy, w, h = nums
                out.append((c, cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2))
            else:
                xs, ys = nums[0::2], nums[1::2]
                out.append((c, min(xs), min(ys), max(xs), max(ys)))
    return out

def predict(paths, conf):
    """{path: (xyxy relatif, cls, conf)}"""
    res_all = {}
    for i in range(0, len(paths), 32):
        for r in model.predict([str(p) for p in paths[i:i + 32]], conf=conf, iou=0.7, imgsz=640,
                               device='0', verbose=False, stream=True):
            b = r.boxes
            res_all[Path(r.path)] = (b.xyxyn.cpu().numpy(), b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy())
    return res_all

## 2. Uji dugaan tumpukan di val (hanya dihitung)

Untuk setiap deteksi yang dianggap **palsu** (conf ≥ 0.25, tidak cocok dengan kotak asli sekelas pada IoU ≥ 0.5):
apakah ia berada **di dalam** kotak asli sekelas (≥ 80% luasnya tertutup)? Kalau iya, kemungkinan besar itu
objek asli yang di label digabung dalam satu kotak tumpukan, bukan kesalahan model.

In [ ]:
def iou_mat(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)

def audit_split(split_dir, conf_fp=0.25, conf_pile=0.5):
    imgs = images(split_dir / 'images')
    preds = predict(imgs, conf_fp)
    fp_rows, pile_rows = [], []
    for p in imgs:
        gt = read_labels(split_dir / 'labels' / f'{p.stem}.txt')
        g_xy = np.array([g[1:] for g in gt]) if gt else np.zeros((0, 4))
        g_c = np.array([g[0] for g in gt], dtype=int)
        pb, pc, ps = preds[p]
        # deteksi palsu: tidak cocok sekelas IoU >= 0.5 (pencocokan serakah per kelas)
        used = set()
        iou = iou_mat(pb, g_xy)
        for i in np.argsort(-ps):
            cand = [j for j in np.where((iou[i] >= 0.5) & (g_c == pc[i]))[0] if j not in used] if len(g_xy) else []
            if cand:
                used.add(max(cand, key=lambda j: iou[i, j]))
                continue
            inside = dv.coverage(pb[i:i + 1], g_xy[g_c == pc[i]]).max() if (g_c == pc[i]).any() else 0.0
            fp_rows.append(dict(img=p.name, kelas=NAMES[pc[i]], conf=float(ps[i]), di_dalam_kotak_sekelas=inside >= 0.8))
        # kotak tumpukan: kotak asli yang memuat >= 2 deteksi yakin sekelas
        strong = ps >= conf_pile
        for j, (c, *box) in enumerate(gt):
            m = strong & (pc == c)
            if m.sum() < 2:
                continue
            cov = dv.coverage(pb[m], np.array([box]))[:, 0]
            n_in = int((cov >= 0.8).sum())
            if n_in >= 2:
                pile_rows.append(dict(img=p.name, idx=j, kelas=NAMES[c], n_objek_di_dalam=n_in,
                                      luas_kotak=(box[2] - box[0]) * (box[3] - box[1]),
                                      conf_min=float(ps[m][cov >= 0.8].min())))
    return pd.DataFrame(fp_rows), pd.DataFrame(pile_rows), preds

fp_val, pile_val, preds_val = audit_split(DATA / 'val')
WEAK = ['turmeric', 'galangal', 'shrimp', 'mushroom', 'cauliflower', 'potato', 'broccoli', 'pumpkin']
uji = fp_val.groupby('kelas').agg(palsu=('img', 'size'), di_dalam_kotak_sekelas=('di_dalam_kotak_sekelas', 'sum'))
uji['persen_di_dalam'] = (uji.di_dalam_kotak_sekelas / uji.palsu * 100).round(1)
uji = uji.reindex([k for k in WEAK if k in uji.index] + [k for k in uji.index if k not in WEAK])
display(uji.head(15))
print(f"Semua kelas: {int(fp_val.di_dalam_kotak_sekelas.sum())} dari {len(fp_val)} deteksi palsu "
      f"({fp_val.di_dalam_kotak_sekelas.mean() * 100:.1f}%) berada di dalam kotak asli sekelas.")
print('Kotak tumpukan di val (dicatat saja):', len(pile_val), '| per kelas:', pile_val.kelas.value_counts().to_dict() if len(pile_val) else {})
uji.to_csv(OUT / 'uji_dugaan_tumpukan_val.csv'); pile_val.to_csv(OUT / 'kotak_tumpukan_val.csv', index=False)

## 3. Kotak tumpukan di train (kandidat perbaikan, belum diubah)

In [ ]:
fp_tr, pile_tr, preds_tr = audit_split(DATA / 'train')
print('Kotak tumpukan di train:', len(pile_tr), 'di', pile_tr.img.nunique() if len(pile_tr) else 0, 'foto')
if len(pile_tr):
    display(pile_tr.groupby('kelas').agg(kotak=('img', 'size'), foto=('img', 'nunique'),
                                         objek_di_dalam=('n_objek_di_dalam', 'sum')).sort_values('kotak', ascending=False))
pile_tr.to_csv(OUT / 'kotak_tumpukan_train.csv', index=False)

def draw(img_path, gt_boxes, pred_boxes, title, size=300):
    im = ImageOps.exif_transpose(Image.open(img_path)).convert('RGB')
    w, h = im.size
    d = ImageDraw.Draw(im)
    lw = max(2, w // 200)
    for b in gt_boxes:      # merah = label asli (kotak tumpukan)
        d.rectangle([b[0] * w, b[1] * h, b[2] * w, b[3] * h], outline=(220, 30, 30), width=lw * 2)
    for b in pred_boxes:    # hijau = deteksi v4b
        d.rectangle([b[0] * w, b[1] * h, b[2] * w, b[3] * h], outline=(30, 200, 60), width=lw)
    im.thumbnail((size, size - 20))
    tile = Image.new('RGB', (size, size), 'white')
    tile.paste(im, ((size - im.width) // 2, 20))
    ImageDraw.Draw(tile).text((4, 4), title[:44], fill='black')
    return tile

def sheet(tiles, title, path, cols=4, size=300):
    if not tiles:
        return
    rows = math.ceil(len(tiles) / cols)
    s = Image.new('RGB', (cols * size, rows * size + 24), (240, 240, 240))
    ImageDraw.Draw(s).text((6, 6), title, fill='black')
    for k, t in enumerate(tiles):
        s.paste(t, ((k % cols) * size, 24 + (k // cols) * size))
    s.save(path, quality=82)

if len(pile_tr):
    sample = pile_tr.sort_values('n_objek_di_dalam', ascending=False).groupby('kelas').head(3).head(16)
    tiles = []
    for r in sample.itertuples():
        p = DATA / 'train/images' / r.img
        gt = read_labels(DATA / 'train/labels' / f'{p.stem}.txt')[r.idx]
        pb, pc, ps = preds_tr[p]
        m = (pc == IDX[r.kelas]) & (ps >= 0.5)
        tiles.append(draw(p, [gt[1:]], pb[m], f'{r.kelas}: 1 kotak, {r.n_objek_di_dalam} objek'))
    sheet(tiles, 'Train: merah = label (satu kotak), hijau = deteksi v4b per objek', OUT / 'review_tumpukan_train.jpg')

## 4. Unduh dan periksa 5 dataset rimpang Roboflow

Pemetaan kelas: jahe → `ginger`, kunyit → `turmeric`, lengkuas → `galangal`. Rimpang lain (kencur, temulawak, dll.)
tidak diberi label, sehingga menjadi contoh negatif. `Wild_turmeric` dianggap ambigu.

In [ ]:
from roboflow import Roboflow
import requests
from kaggle_secrets import UserSecretsClient
KEY = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=KEY)

SOURCES = {
    'soda':   'soda-fj5ov/deteksi-rempah-rempah-2',
    'rival':  'rival-0zxtv/rempah-r4adj',
    'ginger': 'fruits-2kbl9/detection-ginger-t9kqu',
    'rimpang10': 'rimpang-key8c/rempah-c9tpp',
    'ayush':  'ayushvision/ayushvision_tuber',
}
MAP = {'jahe': 'ginger', 'ginger': 'ginger', 'kunyit': 'turmeric', 'turmeric': 'turmeric',
       'lengkuas': 'galangal', 'galangal': 'galangal', 'laos': 'galangal', 'wild_turmeric': 'AMBIGU'}

info, local = {}, {}
for key, slug in SOURCES.items():
    ws, proj = slug.split('/')
    try:
        j = requests.get(f'https://api.roboflow.com/{slug}', params={'api_key': KEY}, timeout=30).json()
        vers = sorted(int(v['id'].split('/')[-1]) for v in j.get('versions', []))
        if not vers:
            info[key] = {'status': 'tidak ada versi yang bisa diunduh'}
            print(key, '-> dilewati: tidak ada versi'); continue
        v = vers[-1]
        dest = TMP / 'rimpang' / key
        rf.workspace(ws).project(proj).version(v).download('yolov8', location=str(dest), overwrite=True)
        local[key] = dest
        info[key] = {'status': 'ok', 'versi': v, 'versi_tersedia': vers}
        print(key, '-> versi', v, 'diunduh')
    except Exception as e:
        info[key] = {'status': f'gagal: {e}'}
        print(key, '-> gagal:', e)
del KEY

In [ ]:
def ds_items(root):
    cfg = yaml.safe_load((root / 'data.yaml').read_text())
    names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][k] for k in sorted(cfg['names'])]
    items = []
    for split in ('train', 'valid', 'test'):
        d = root / split / 'images'
        if not d.is_dir():
            continue
        for p in sorted(d.iterdir()):
            if p.suffix.lower() in IMG_EXT:
                lbl = root / split / 'labels' / f'{p.stem}.txt'
                items.append(dict(split=split, img=p, boxes=read_labels(lbl)))
    return names, items

# hash data ReSnap (train/val v4 + test lama hanya untuk cek foto kembar, tidak dievaluasi)
ref = {}
for split_dir in [DATA / 'train', DATA / 'val'] + ([BASE / 'test'] if BASE else []):
    ref.update({f'resnap/{split_dir.name}/{Path(k).name}': h for k, h in dv.hash_folder(split_dir / 'images').items()})
print('Hash referensi ReSnap:', len(ref))

stats, hashes = {}, {}
for key, root in local.items():
    names, items = ds_items(root)
    mapped = {i: MAP.get(n.strip().lower().replace(' ', '_').replace('-', '_'), 'negatif') for i, n in enumerate(names)}
    box_cls = Counter(); big = 0; per_img = []
    for it in items:
        per_img.append(len(it['boxes']))
        for c, x1, y1, x2, y2 in it['boxes']:
            box_cls[mapped.get(c, '?')] += 1
            big += (x2 - x1) * (y2 - y1) >= 0.5
    h = {str(it['img']): dv.dhash(it['img']) for it in items}
    h = {k: v for k, v in h.items() if v is not None}
    hashes[key] = h
    # salinan augmentasi: foto dalam dataset yang sama yang hampir identik
    keys = list(h); arr = np.array([h[k] for k in keys], dtype=np.uint64)
    groups = 0; seen = np.zeros(len(keys), bool)
    for i in range(len(keys)):
        if seen[i]:
            continue
        x = np.bitwise_xor(arr, arr[i]); dist = np.unpackbits(x.view(np.uint8).reshape(-1, 8), axis=1).sum(1)
        seen |= dist <= 4; groups += 1
    kembar_resnap = dv.near_duplicates(h, ref, max_dist=6)
    stats[key] = dict(foto=len(items), per_split=dict(Counter(it['split'] for it in items)),
                      kelas_asli=names, pemetaan=mapped, kotak_per_kelas=dict(box_cls),
                      kotak_per_foto_median=float(np.median(per_img)) if per_img else 0,
                      kotak_raksasa_persen=round(100 * big / max(1, sum(box_cls.values())), 1),
                      foto_unik_kira2=groups, kembar_dengan_resnap=len(kembar_resnap))
    print(f"{key}: {len(items)} foto, ~{groups} unik, kotak {dict(box_cls)}, kembar dgn ReSnap {len(kembar_resnap)}")

# foto kembar antar-dataset rimpang (penting untuk memilih test eksternal)
antar = {}
ks = list(hashes)
for a in ks:
    for b in ks:
        if a < b:
            antar[f'{a}~{b}'] = len(dv.near_duplicates(hashes[a], hashes[b], max_dist=6))
print('Foto kembar antar-dataset:', antar)

## 5. Bahan ReSnap lain yang tidak berlabel, dan lembar pratinjau

In [ ]:
RIMPANG_IDS = {IDX['ginger'], IDX['turmeric'], IDX['galangal']}
for key, root in local.items():
    names, items = ds_items(root)
    sample = random.sample(items, min(300, len(items)))
    preds = predict([it['img'] for it in sample], 0.6)
    lain = Counter()
    for it in sample:
        pb, pc, ps = preds[it['img']]
        gt = np.array([b[1:] for b in it['boxes']]) if it['boxes'] else np.zeros((0, 4))
        cov = dv.coverage(pb, gt).max(1) if len(gt) else np.zeros(len(pb))
        found = {NAMES[c] for c, o in zip(pc, cov) if c not in RIMPANG_IDS and o < 0.5}
        for f in found:
            lain[f] += 1
    stats[key]['bahan_lain_tak_berlabel_di_300_foto'] = dict(lain.most_common(8))
    stats[key]['foto_dengan_bahan_lain_persen'] = round(100 * sum(
        1 for it in sample if any(c not in RIMPANG_IDS and o < 0.5 for c, o in zip(
            preds[it['img']][1], (dv.coverage(preds[it['img']][0], np.array([b[1:] for b in it['boxes']])).max(1)
                                  if it['boxes'] else np.zeros(len(preds[it['img']][1])))))) / len(sample), 1)
    # pratinjau 12 foto dengan label yang sudah dipetakan
    tiles = []
    for it in random.sample(items, min(12, len(items))):
        im = ImageOps.exif_transpose(Image.open(it['img'])).convert('RGB'); w, h = im.size
        d = ImageDraw.Draw(im); lw = max(2, w // 150)
        col = {'ginger': (240, 180, 0), 'turmeric': (230, 90, 0), 'galangal': (200, 0, 120), 'negatif': (120, 120, 120), 'AMBIGU': (0, 120, 255)}
        for c, x1, y1, x2, y2 in it['boxes']:
            lab = stats[key]['pemetaan'].get(c, '?')
            d.rectangle([x1 * w, y1 * h, x2 * w, y2 * h], outline=col.get(lab, (0, 0, 0)), width=lw)
        im.thumbnail((300, 280))
        t = Image.new('RGB', (300, 300), 'white'); t.paste(im, ((300 - im.width) // 2, 20))
        ImageDraw.Draw(t).text((4, 4), f"{it['split']} | {len(it['boxes'])} kotak", fill='black')
        tiles.append(t)
    sheet(tiles, f'{key}: kuning=jahe, oranye=kunyit, ungu=lengkuas, abu=negatif (tak dilabel), biru=ambigu',
          OUT / f'pratinjau_{key}.jpg')
    # simpan data mentah untuk dijadikan dataset Kaggle
    shutil.make_archive(str(RAW / key), 'zip', root)
    print(key, 'bahan lain tak berlabel:', stats[key]['bahan_lain_tak_berlabel_di_300_foto'])

ringkas = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk not in ('kelas_asli', 'pemetaan')} for k, v in stats.items()}).T
display(ringkas)

## 6. Paket hasil

In [ ]:
summary = {
    'uji_dugaan_tumpukan_val': {
        'deteksi_palsu': int(len(fp_val)),
        'di_dalam_kotak_sekelas': int(fp_val.di_dalam_kotak_sekelas.sum()),
        'per_kelas_lemah': uji.loc[[k for k in WEAK if k in uji.index]].to_dict(orient='index'),
    },
    'kotak_tumpukan': {'val': int(len(pile_val)), 'train': int(len(pile_tr)),
                       'train_per_kelas': pile_tr.kelas.value_counts().to_dict() if len(pile_tr) else {}},
    'rimpang_unduh': info,
    'rimpang_statistik': stats,
    'rimpang_kembar_antar_dataset': antar,
}
(OUT / 'ringkasan.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str))
zp = OUTPUT / 'audit_v5.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        z.write(f, f.name)
print(json.dumps(summary, indent=1, ensure_ascii=False, default=str)[:4000])
print('Laporan:', zp, round(zp.stat().st_size / 1e6, 2), 'MB')
print('Data mentah rimpang:', [(p.name, round(p.stat().st_size / 1e6)) for p in RAW.iterdir()])